# 02 — Topology Visualisation: Ring vs Elevator

Visualises the two topologies used in experiments:
- **Ring** (Condition A baseline) — static, no healing
- **Elevator** (HEAL) — hub-and-spoke, diameter=2, self-healing

In [ ]:
import subprocess, sys, os

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "src")):
    repo_root = os.path.dirname(repo_root)

assert os.path.isdir(os.path.join(repo_root, "src")), \
    f"src/ not found. cwd={os.getcwd()} — open this notebook from the fl-rockstar repo root"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "datasets", "networkx",
    "numpy", "scipy", "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml", "--quiet"])

import torch
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.topology import build_elevator_topology, build_ring_topology, heal_recover_hubs

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"device: {device}  |  root: {repo_root}")

In [ ]:
N = 30  # smaller n for clear visualisation
ring_G, _ = build_ring_topology(N)
elev_G, hub_ids = build_elevator_topology(N, n_hubs=3, seed=42)
print(f'Ring:     nodes={ring_G.number_of_nodes()}, edges={ring_G.number_of_edges()}')
print(f'Elevator: nodes={elev_G.number_of_nodes()}, edges={elev_G.number_of_edges()}')
print(f'Hub IDs: {hub_ids}')
print(f'Elevator diameter: {nx.diameter(elev_G)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ring
ring_pos = nx.circular_layout(ring_G)
nx.draw_networkx(ring_G, ring_pos, ax=axes[0], node_size=120,
                 node_color='#94A3B8', with_labels=False, edge_color='#CBD5E1')
axes[0].set_title('Ring Topology (Condition A — No Healing)', fontsize=13)
axes[0].axis('off')

# Elevator
elev_pos = nx.spring_layout(elev_G, seed=42, k=2)
node_colors = ['#0284C7' if n in hub_ids else '#BAE6FD' for n in elev_G.nodes()]
nx.draw_networkx_nodes(elev_G, elev_pos, ax=axes[1], node_size=200,
                       node_color=node_colors)
nx.draw_networkx_edges(elev_G, elev_pos, ax=axes[1], edge_color='#E2E8F0', alpha=0.6)
nx.draw_networkx_labels(elev_G, elev_pos, ax=axes[1], font_size=6, font_color='white',
                        labels={n: 'H' if n in hub_ids else '' for n in elev_G.nodes()})

hub_patch   = mpatches.Patch(color='#0284C7', label='Hub node')
client_patch = mpatches.Patch(color='#BAE6FD', label='Client node')
axes[1].legend(handles=[hub_patch, client_patch], loc='lower right')
axes[1].set_title('Elevator Topology (HEAL — Self-Healing)', fontsize=13)
axes[1].axis('off')

plt.suptitle('Topology Comparison: Ring vs Elevator', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/02_topology_comparison.png', dpi=150)
plt.show()

In [ ]:
# Simulate hub failure and re-election
import copy
G_after = copy.deepcopy(elev_G)
failed = hub_ids[:1]  # kill one hub
print(f'Before: hubs={hub_ids}')

G_after, new_hub_ids = heal_recover_hubs(G_after, list(hub_ids), failed, n_hubs=3)
print(f'After killing hub {failed}: new_hubs={new_hub_ids}')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (G, h_ids, title) in zip(axes, [
    (elev_G, hub_ids, 'Before Failure'),
    (G_after, new_hub_ids, f'After Hub {failed[0]} Fails → Self-Healed'),
]):
    pos = nx.spring_layout(G, seed=42)
    colors = ['#EF4444' if n in failed else ('#0284C7' if n in h_ids else '#BAE6FD') for n in G.nodes()]
    nx.draw_networkx(G, pos, ax=ax, node_size=200, node_color=colors,
                     with_labels=False, edge_color='#E2E8F0', alpha=0.7)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('HEAL Self-Healing: Hub Re-Election', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/02_heal_recovery.png', dpi=150)
plt.show()